# 01 · Data audit, module selection, outcomes and splits
Generated by `notebooks/make_notebooks.py`; all numbers come from `scripts/01_preprocess.py` outputs.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
P = ROOT / 'data' / 'processed'; TAB = ROOT / 'results' / 'tables'; FIG = ROOT / 'results' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)


## Raw table audit (columns, dtypes, missing values)

In [2]:
audit = json.load(open(TAB/'audit_raw_tables.json'))
for k in ['courses','assessments','studentAssessment','vle','studentRegistration','studentInfo']:
    print(k, audit[k]['rows'], 'rows; missing:', audit[k]['missing'])
print('presentations:', audit['presentations'])

courses 22 rows; missing: {}
assessments 206 rows; missing: {'date': 11}
studentAssessment 173912 rows; missing: {'score': 173}
vle 6364 rows; missing: {'week_from': 5243, 'week_to': 5243}
studentRegistration 32593 rows; missing: {'date_registration': 45, 'date_unregistration': 22521}
studentInfo 32593 rows; missing: {'imd_band': 1111}
presentations: ['2013B', '2013J', '2014B', '2014J']


## Activity types (voluntary set V is checked against these)

In [3]:
at = pd.read_csv(TAB/'audit_vle_activity_types.csv')
at.pivot_table(index='activity_type', columns='code_module', values='n_sites', aggfunc='sum').fillna(0).astype(int)

code_module,AAA,BBB,CCC,DDD,EEE,FFF,GGG
activity_type,,,,,,,
dataplus,8,0,0,0,0,20,0
dualpane,0,0,0,0,3,17,0
externalquiz,0,0,0,26,0,0,0
folder,0,0,0,0,0,2,0
forumng,21,56,18,53,15,25,6
glossary,4,5,0,5,0,4,3
homepage,2,4,2,4,3,4,3
htmlactivity,0,0,0,0,0,4,0
oucollaborate,4,6,4,14,7,47,0


## Non-exam assessments per module presentation and module selection

In [4]:
ne = pd.read_csv(TAB/'audit_non_exam_per_presentation.csv')
display(ne.pivot(index='code_module', columns='code_presentation', values='n_non_exam'))
display(ne.pivot(index='code_module', columns='code_presentation', values='n_distinct_dates'))
pd.read_csv(TAB/'audit_module_selection.csv')

code_presentation,2013B,2013J,2014B,2014J
code_module,,,,
AAA,NaN,5.0,NaN,5.0
BBB,11.0,11.0,11.0,5.0
CCC,NaN,NaN,8.0,8.0
DDD,13.0,6.0,6.0,6.0
EEE,NaN,4.0,4.0,4.0
FFF,12.0,12.0,12.0,12.0
GGG,NaN,9.0,9.0,9.0


code_presentation,2013B,2013J,2014B,2014J
code_module,,,,
AAA,NaN,5.0,NaN,5.0
BBB,7.0,7.0,7.0,5.0
CCC,NaN,NaN,8.0,8.0
DDD,13.0,6.0,6.0,6.0
EEE,NaN,4.0,4.0,4.0
FFF,6.0,6.0,6.0,6.0
GGG,NaN,4.0,4.0,4.0


,code_module,presentations,all_ok_count,all_ok_topics,min_topics,max_topics,has_train,has_val,has_test,selected
0,AAA,"2013J,2014J",True,True,5,5,True,False,True,False
1,BBB,"2013B,2013J,2014B,2014J",True,True,5,7,True,True,True,False
2,CCC,"2014B,2014J",True,True,8,8,False,True,True,False
3,DDD,"2013B,2013J,2014B,2014J",True,True,6,13,True,True,True,True
4,EEE,"2013J,2014B,2014J",False,False,4,4,True,True,True,False
5,FFF,"2013B,2013J,2014B,2014J",True,True,6,6,True,True,True,True
6,GGG,"2013J,2014B,2014J",True,False,4,4,True,True,True,False


In [5]:
fig, ax = plt.subplots(figsize=(7,3.5))
pv = ne.pivot(index='code_module', columns='code_presentation', values='n_distinct_dates')
pv.plot.bar(ax=ax)
ax.axhline(5, color='k', ls='--', lw=1, label='minimum 5 topics')
ax.set_ylabel('distinct non-exam due dates (= topics)'); ax.set_title('Topics per module presentation after merging same-day assessments')
ax.legend(fontsize=7, ncol=3); fig.tight_layout(); fig.savefig(FIG/'eda_topics_per_presentation.png', dpi=150)

## Topic windows (selected modules)

In [6]:
topics = pd.read_parquet(P/'topics.parquet'); topics.drop(columns='assessment_ids')

,code_module,code_presentation,topic,t_open,t_due,assessment_types,n_assessments,weight,n_topics
0,DDD,2013B,1,-17.0,25.0,TMA,1,7.5,6
1,DDD,2013B,2,25.0,53.0,TMA,1,10.0,6
2,DDD,2013B,3,53.0,81.0,TMA,1,12.5,6
3,DDD,2013B,4,81.0,116.0,TMA,1,15.0,6
4,DDD,2013B,5,116.0,151.0,TMA,1,15.0,6
5,DDD,2013B,6,151.0,200.0,TMA,1,15.0,6
6,DDD,2013J,1,-19.0,25.0,TMA,1,10.0,6
7,DDD,2013J,2,25.0,53.0,TMA,1,12.5,6
8,DDD,2013J,3,53.0,88.0,TMA,1,17.5,6
9,DDD,2013J,4,88.0,123.0,TMA,1,20.0,6


## id_site → topic mapping: share mapped by week_from vs. cohort clicks

In [7]:
pd.read_csv(TAB/'topic_map_method_share.csv')

,code_module,map_method,n_sites,clicks,share_sites,share_clicks
0,DDD,cohort_clicks,1666,4023415.8,0.975410,0.739239
1,DDD,excluded_navigation,4,1409694.0,0.002342,0.259009
2,DDD,unmapped_no_clicks,36,0.0,0.021077,0.000000
3,DDD,week_from,2,9536.0,0.001171,0.001752
4,FFF,cohort_clicks,1120,10347892.0,0.569395,0.597501
5,FFF,excluded_navigation,4,2379013.0,0.002034,0.137367
6,FFF,unmapped_no_clicks,17,0.0,0.008643,0.000000
7,FFF,week_from,826,4591720.0,0.419929,0.265132


## Outcomes

In [8]:
inter = pd.read_parquet(P/'interactions.parquet')
print(json.load(open(TAB/'audit_cleaning_log.json')))
inter.groupby(['code_module','split']).agg(rows=('y','size'), submitted=('is_submitted','mean'), pass_rate=('y','mean'), mean_score=('score','mean')).round(3)

{'interactions': {'rows_all_student_topic': 84204, 'dropped_after_withdrawal': 19633, 'dropped_banked': 949, 'dropped_no_score': 82, 'rows_final': 63540, 'share_submitted': 0.7956248032735285, 'pass_rate_submitted': 0.9418245836135617}, 'studentAssessment': {'rows_raw': 173912, 'banked': 1909, 'missing_score': 172, 'rows_clean': 171831}}


rows  submitted  pass_rate  mean_score
code_module split                                         
DDD         test    7882      0.852      0.935      72.693
            train  14819      0.760      0.902      69.893
            val     5395      0.729      0.893      69.348
FFF         test    9888      0.836      0.971      77.247
            train  18618      0.808      0.963      75.490
            val     6938      0.771      0.965      75.042

In [9]:
fig, axes = plt.subplots(1,2, figsize=(10,3.5))
for ax, m in zip(axes, ['DDD','FFF']):
    d = inter[(inter.code_module==m)&inter.score.notna()]
    ax.hist(d.score, bins=50, color='#4c72b0'); ax.axvline(40, color='r', ls='--', label='pass mark 40'); ax.axvline(70, color='orange', ls=':', label='70 (sensitivity)')
    ax.set_title(f'{m}: topic assessment score'); ax.set_xlabel('score'); ax.set_ylabel('student-topics'); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG/'eda_score_hist.png', dpi=150)

## Splits

In [10]:
pd.read_csv(TAB/'split_counts.csv')

,code_module,code_presentation,split,n_students
0,DDD,2013B,train,1303
1,DDD,2013J,train,1938
2,DDD,2014B,val,1228
3,DDD,2014J,test,1803
4,FFF,2013B,train,1614
5,FFF,2013J,train,2283
6,FFF,2014B,val,1500
7,FFF,2014J,test,2365


In [11]:
stu = pd.read_parquet(P/'students.parquet')
stu.groupby(['code_module','split']).agg(n=('sid','size'), exam_taken=('exam_score', lambda s: s.notna().mean()), exam_mean=('exam_score','mean')).round(3)

n  exam_taken  exam_mean
code_module split                             
DDD         test   1803       0.527     63.308
            train  3241       0.484     61.693
            val    1228       0.427     69.101
FFF         test   2365       0.000        NaN
            train  3897       0.000        NaN
            val    1500       0.000        NaN